<a href="https://colab.research.google.com/github/SebastianSanchez5/DataAnalysis/blob/main/Assignment2_TimeSeries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Task 1: Setup

Install the required Python packages: pandas, matplotlib, seaborn, and fredapi.
Obtain an API key from FRED (https://fred.stlouisfed.org/).


In [ ]:
#@title Setup

!pip install -U -q fredapi

from fredapi import Fred
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from statsmodels.tsa.seasonal import seasonal_decompose

from prophet import Prophet
from prophet.plot import plot_plotly, plot_components_plotly, plot_cross_validation_metric
from prophet.diagnostics import cross_validation, performance_metrics

In [ ]:
#@title Plotting Setup

%config InlineBackend.figure_format = 'retina'

# Change the graph defaults
plt.rcParams['figure.figsize'] = (8, 3)  # Default figure size of 6x2 inches
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = 'lightgray'
plt.rcParams['font.size'] = 10  # Default font size of 12 points
plt.rcParams['lines.linewidth'] = 1  # Default line width of 1 points
plt.rcParams['lines.markersize'] = 3  # Default marker size of 3 points
plt.rcParams['legend.fontsize'] = 10  # Default legend font size of 10 points

In [ ]:
fred_api_key='2d84d7ea6297f9d198a5575560946e1a'

fred = Fred(api_key=fred_api_key)

## Task 2: Data Retrieval

Use the fredapi Python package to fetch data for these economic indicators

* Inflation Rate (Consumer Price Index) -- CPIAUCSL
* Real Gross Domestic Product (GDP) -- GDPC1
* Unemployment Rate -- UNRATE
* Federal Funds Rate -- FEDFUNDS
* Advance Retail Sales: Retail Trade -- RSXFSN
* Housing Starts -- HOUSTNSA

Plot the 6 time series. Add a title, labels for the x and y axes, and a legend to each plot.


In [ ]:
# prompt: se the fredapi Python package to fetch data for these economic indicators
# Inflation Rate (Consumer Price Index) -- CPIAUCSL
# Real Gross Domestic Product (GDP) -- GDPC1
# Unemployment Rate -- UNRATE
# Federal Funds Rate -- FEDFUNDS
# Advance Retail Sales: Retail Trade -- RSXFSN
# Housing Starts -- HOUSTNSA
# Plot the 6 time series. Add a title, labels for the x and y axes, and a legend to each plot.

import matplotlib.pyplot as plt
# Dictionary of indicators and their FRED series IDs
indicators = {
    'Inflation Rate (CPI)': 'CPIAUCSL',
    'Real GDP': 'GDPC1',
    'Unemployment Rate': 'UNRATE',
    'Federal Funds Rate': 'FEDFUNDS',
    'Retail Sales': 'RSXFSN',
    'Housing Starts': 'HOUSTNSA'
}

# Fetch data for each indicator and store in a dictionary of DataFrames
data = {}
for name, series_id in indicators.items():
    data[name] = fred.get_series(series_id)

# Plot each time series
for name, series_data in data.items():
    plt.figure()
    series_data.plot()
    plt.title(name)
    plt.xlabel('Date')
    plt.ylabel('Value')
    plt.legend([name])
    plt.show()

## Task 3: Seasonality Decomposition and Normalization


1. For  the `Retail Sales` and `Housing Starts` series, observe their seasonality. The seasonality should be clearly visible in the plot.
Decompose the `Retail Sales` and `Housing Starts` series into their corresponding trend, seasonal, and residual components using a method of your choice.

  Remove the seasonality from the time series: the time series without the seasonality contains the trend and the residuals.

2. Normalize the de-seasonalized `Retail Sales` time series to account for inflation. Convert all the dollar amounts to Jan-2012 dollars, to be comparable with the GDP time series, which is also expressed in 2012 dollars.

3. Using the CPI time series, create a new time series `inflation` that shows the yearly inflation, which is the percentage change of CPI from twelve months ago.

4. Using the GDP time series, create a new time series `gdp_growth` showing the yearly growth of the GDP.

5. For the GDP and GDP growth time series with quarterly observations, convert them to monthly, using the `resample` command. Fill in the missing values using the `fillna()` command using the forward filling method (`ffill`), which fills in the missing values with the latest known value at the time.

6. Plot the time series:
* Deseasonalized retail sales, adjusted for inflation
* Deseasonalized housing starts
* Year-over-Year percentage change of the CPI
* Year-over-Year percentage change of the GDP

  Add a title, labels for the x and y axes, and a legend to each plot.

In [ ]:
# prompt: For the Retail Sales and Housing Starts series, observe their seasonality. The seasonality should be clearly visible in the plot. Decompose the Retail Sales and Housing Starts series into their corresponding trend, seasonal, and residual components using a method of your choice.
# Remove the seasonality from the time series: the time series without the seasonality contains the trend and the residuals.

import matplotlib.pyplot as plt
# Function to perform seasonal decomposition and plot
def plot_seasonal_decomposition(series, title):
    plt.figure(figsize=(12, 8))
    result = seasonal_decompose(series, model='additive')
    result.plot()
    plt.suptitle(title + ' - Seasonal Decomposition', y=1.02)
    plt.show()
    return result

# Decompose Retail Sales
retail_sales = data['Retail Sales'].dropna()
retail_sales_decomposition = plot_seasonal_decomposition(retail_sales, 'Retail Sales')

# Decompose Housing Starts
housing_starts = data['Housing Starts'].dropna()
housing_starts_decomposition = plot_seasonal_decomposition(housing_starts, 'Housing Starts')

# Remove seasonality from Retail Sales and Housing Starts
deseasonalized_retail_sales = retail_sales - retail_sales_decomposition.seasonal
deseasonalized_housing_starts = housing_starts - housing_starts_decomposition.seasonal

# Plot deseasonalized series (optional, to visualize the result of deseasoning)
plt.figure()
deseasonalized_retail_sales.plot()
plt.title('Deseasonalized Retail Sales')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend(['Deseasonalized Retail Sales'])
plt.show()

plt.figure()
deseasonalized_housing_starts.plot()
plt.title('Deseasonalized Housing Starts')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend(['Deseasonalized Housing Starts'])
plt.show()

In [ ]:

# Get CPI series
cpi = data['Inflation Rate (CPI)'].dropna()

# Find the CPI value in Jan 2012
cpi_jan_2012 = cpi.loc['2012-01-01']

# Normalize the deseasonalized retail sales using the Jan 2012 CPI value
# The formula for converting to a base period is:
# Value_in_Base_Period_Dollars = (Value_in_Current_Dollars / CPI_in_Current_Period) * CPI_in_Base_Period
# Here, the base period is Jan 2012.
deseasonalized_retail_sales_adjusted = (deseasonalized_retail_sales / cpi) * cpi_jan_2012

# Align the indices to ensure correct division (Retail Sales is monthly, CPI is monthly)
# Ensure both series cover the same time period if needed, though fredapi usually aligns well
# If issues arise, you might need to resample/align the series explicitly.
# For this specific case, both are typically monthly series from FRED, so direct division should work after dropping NaNs.
# Let's ensure the division is done on the overlapping dates:
common_index = deseasonalized_retail_sales_adjusted.dropna().index
deseasonalized_retail_sales_adjusted = deseasonalized_retail_sales_adjusted[common_index]

# Print the first few values of the normalized series to check
print("Normalized Deseasonalized Retail Sales (Jan 2012 USD):")
print(deseasonalized_retail_sales_adjusted.head())

In [ ]:

# Calculate yearly inflation (12-month percentage change in CPI)
inflation = cpi.pct_change(periods=12) * 100

In [ ]:

import matplotlib.pyplot as plt
# Get GDP series
gdp = data['Real GDP'].dropna()

# Calculate yearly GDP growth (4-quarter percentage change in GDP)
# GDP data from FRED (GDPC1) is quarterly, so use periods=4 for yearly growth
gdp_growth = gdp.pct_change(periods=4) * 100

# Resample GDP and GDP growth to monthly frequency using forward fill
gdp_monthly = gdp.resample('M').ffill()
gdp_growth_monthly = gdp_growth.resample('M').ffill()

# Plot the required time series
plt.figure()
deseasonalized_retail_sales_adjusted.plot()
plt.title('Deseasonalized Retail Sales (Adjusted for Inflation, Jan 2012 USD)')
plt.xlabel('Date')
plt.ylabel('Value (Jan 2012 USD)')
plt.legend(['Deseasonalized Retail Sales'])
plt.show()

plt.figure()
deseasonalized_housing_starts.plot()
plt.title('Deseasonalized Housing Starts')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend(['Deseasonalized Housing Starts'])
plt.show()

plt.figure()
inflation.plot()
plt.title('Year-over-Year CPI Inflation')
plt.xlabel('Date')
plt.ylabel('Percentage Change (%)')
plt.legend(['CPI Inflation'])
plt.show()

plt.figure()
gdp_growth_monthly.plot()
plt.title('Year-over-Year GDP Growth (Monthly, Forward-Filled)')
plt.xlabel('Date')
plt.ylabel('Percentage Change (%)')
plt.legend(['GDP Growth'])
plt.show()

## Task 4: Correlation Analysis

1. Create a dataframe with the following time series:

* Inflation (yearly change of the CPI series)
* GDP growth (yearly change of the GDP)
* Unemployment rate
* Federal Funds rate
* Deseasonalized housing starts
* Deseasonalized and deflated retail sales

  You can use the [`pandas.concat`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html) function to put all the time series together in one dataframe, using a code similar to the one below:

  ```
  df = pd.concat([inflation, gdp_growth, unemp, fedrates, housing_deseasonalized, retail_deseasonalized_deflated], axis="columns")
  df.columns = ["INFLATION", "GDP_GROWTH", "UNEMPLOYMENT", "FEDRATE", "HOUSING", "RETAIL"]
  ```

2. Using a scatterplot, plot the following:
  * INFLATION vs FEDRATE
  * GDP_GROWTH vs UNEMPLOYMENT
  * HOUSING vs UNEMPLOYMENT
  * HOUSING vs GDP_GROWTH
  * RETAIL vs FEDRATE

3. Define the variable `DECADE` using the command

  ```
  df['DECADE']  = 10 * (df.index.year // 10)
  ```
  
  Plot the timeseries `RETAIL` vs `UNEMPLOYMENT` and use color to differentiate the decades using the option `c="DECADE"` in `pd.plot()`. Use the `cmap` option to select the palette for the color.



3. Using the [`pandas.corr()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html) command and the [`seaborn.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html) create and visualize a correlation matrix to check for any correlations between the four series.

4. Experiment with the [`seaborn.pairplot`](https://seaborn.pydata.org/generated/seaborn.pairplot.html) and explore the pairwise relationships across the six time series.



In [ ]:
# prompt: Create a dataframe with the following time series:
# Inflation (yearly change of the CPI series)
# GDP growth (yearly change of the GDP)
# Unemployment rate
# Federal Funds rate
# Deseasonalized housing starts
# Deseasonalized and deflated retail sales

import pandas as pd
# Align the series before concatenating. Resample to 'M' and use ffill.
# Some series are already monthly, others quarterly or need transformation.
# Ensure all series are aligned to a monthly frequency for consistent time series analysis.

# Inflation (already monthly % change)
inflation_aligned = inflation.resample('M').ffill()

# GDP growth (already monthly forward-filled)
gdp_growth_aligned = gdp_growth_monthly

# Unemployment rate (already monthly)
unemp = data['Unemployment Rate'].dropna()
unemp_aligned = unemp.resample('M').ffill()

# Federal Funds rate (already monthly)
fedrates = data['Federal Funds Rate'].dropna()
fedrates_aligned = fedrates.resample('M').ffill()

# Deseasonalized housing starts (already monthly after deseasoning, potentially needs ffill if NaNs introduced)
housing_deseasonalized = deseasonalized_housing_starts.resample('M').ffill()

# Deseasonalized and deflated retail sales (already monthly after transformation, potentially needs ffill if NaNs introduced)
retail_deseasonalized_deflated = deseasonalized_retail_sales_adjusted.resample('M').ffill()


# Concatenate the aligned time series into a single DataFrame
df = pd.concat([
    inflation_aligned,
    gdp_growth_aligned,
    unemp_aligned,
    fedrates_aligned,
    housing_deseasonalized,
    retail_deseasonalized_deflated
], axis="columns")

# Rename the columns
df.columns = ["INFLATION", "GDP_GROWTH", "UNEMPLOYMENT", "FEDRATE", "HOUSING", "RETAIL"]

# Drop rows with any NaN values that resulted from misalignment before the earliest common date
df.dropna(inplace=True)

# Print the first few rows of the created dataframe to verify
print("\nCreated DataFrame:")
print(df.head())

In [ ]:
# prompt: Using a scatterplot, plot the following from above:
# INFLATION vs FEDRATE
# GDP_GROWTH vs UNEMPLOYMENT
# HOUSING vs UNEMPLOYMENT
# HOUSING vs GDP_GROWTH
# RETAIL vs FEDRATE

import matplotlib.pyplot as plt
# 2. Using a scatterplot, plot the following:
#   * INFLATION vs FEDRATE
#   * GDP_GROWTH vs UNEMPLOYMENT
#   * HOUSING vs UNEMPLOYMENT
#   * HOUSING vs GDP_GROWTH
#   * RETAIL vs FEDRATE

# Define the pairs to plot
scatter_pairs = [
    ('INFLATION', 'FEDRATE'),
    ('GDP_GROWTH', 'UNEMPLOYMENT'),
    ('HOUSING', 'UNEMPLOYMENT'),
    ('HOUSING', 'GDP_GROWTH'),
    ('RETAIL', 'FEDRATE')
]

# Plot each scatterplot
for x_col, y_col in scatter_pairs:
    plt.figure()
    plt.scatter(df[x_col], df[y_col], alpha=0.6, s=10) # s controls marker size
    plt.title(f'{y_col} vs {x_col}')
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.grid(True)
    plt.show()

In [ ]:
# prompt: Define the variable DECADE using the command
# df['DECADE']  = 10 * (df.index.year // 10)
# Plot the timeseries RETAIL vs UNEMPLOYMENT and use color to differentiate the decades using the option c="DECADE" in pd.plot(). Use the cmap option to select the palette for the color.

import matplotlib.pyplot as plt
df['DECADE']  = 10 * (df.index.year // 10)

plt.figure(figsize=(10, 6))
df.plot.scatter(x='UNEMPLOYMENT', y='RETAIL', c='DECADE', cmap='viridis', s=10)
plt.title('Retail Sales vs Unemployment Rate by Decade')
plt.xlabel('Unemployment Rate (%)')
plt.ylabel('Retail Sales (Deseasonalized, Adjusted for Inflation, Jan 2012 USD)')
plt.show()

In [ ]:
# prompt: Using the pandas.corr() command and the seaborn.heatmap create and visualize a correlation matrix to check for any correlations between the four series.

import matplotlib.pyplot as plt
# 3. Using the pandas.corr() command and the seaborn.heatmap create and visualize a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Economic Indicators')
plt.show()

In [ ]:
# prompt: Experiment with the seaborn.pairplot and explore the pairwise relationships across the six time series.

import matplotlib.pyplot as plt
# 4. Experiment with the seaborn.pairplot and explore the pairwise relationships across the six time series.
# Use the created dataframe 'df' which contains the six time series.
# The 'DECADE' column can be used as a hue to differentiate points by decade if desired,
# but for a general overview of pairwise relationships, plotting without hue is common.

# Using seaborn.pairplot
# This will create a grid of plots: scatterplots for every pair of columns
# and histograms or kernel density estimates on the diagonal.
sns.pairplot(df.drop(columns=['DECADE'])) # Exclude 'DECADE' for the main pairplot
plt.suptitle('Pairwise Relationships of Economic Indicators', y=1.02) # Add a title to the entire plot
plt.show()

# Optional: Pairplot with hue for 'DECADE' to see relationships across decades
sns.pairplot(df, hue='DECADE', palette='viridis')
plt.suptitle('Pairwise Relationships by Decade', y=1.02)
plt.show()

## Task 5: Modeling and Forecasting

Take the `Advance Retail Sales: Retail Trade -- RSXFSN` time series from FRED. Adjust the values for inflation to May-2024 dollars (using the same process as in Task 3.2) but do not use the statsmodel library for the seasonal decomposition. Insted, use Prophet to extract the seasonal component, and make a forecast for the value  for the next 24 months. Using MAPE and cross-validation, estimate how accurate the forecasts will be.

In [ ]:
# prompt: Take the Advance Retail Sales: Retail Trade -- RSXFSN time series from FRED. Adjust the values for inflation to May-2024 dollars (using the same process as in Task 3.2) but do not use the statsmodel library for the seasonal decomposition. Insted, use Prophet to extract the seasonal component, and make a forecast for the value for the next 24 months. Using MAPE and cross-validation, estimate how accurate the forecasts will be.

import matplotlib.pyplot as plt
# Fetch the original Retail Sales series (RSXFSN)
retail_sales_orig = fred.get_series('RSXFSN').dropna()

# Fetch the CPI series (CPIAUCSL) for inflation adjustment
cpi_orig = fred.get_series('CPIAUCSL').dropna()

# Align the two series by date index
# Find the common time period for both series
common_index = retail_sales_orig.index.intersection(cpi_orig.index)
retail_sales_aligned = retail_sales_orig[common_index]
cpi_aligned = cpi_orig[common_index]

# Get the CPI value for May 2024
# Ensure May 2024 is available in the CPI series. If not, use the latest available month.
try:
    cpi_may_2024 = cpi_aligned.loc['2024-05-01']
except KeyError:
    print("May 2024 CPI data not available. Using the latest available CPI data for inflation adjustment.")
    cpi_may_2024 = cpi_aligned[-1] # Use the last available CPI value
    print(f"Using CPI value from {cpi_aligned.index[-1].strftime('%Y-%m')} for inflation adjustment.")


# Adjust Retail Sales values to May 2024 dollars
# Formula: Value_in_May2024_Dollars = (Value_in_Current_Dollars / CPI_in_Current_Period) * CPI_in_May2024_Period
retail_sales_may2024_dollars = (retail_sales_aligned / cpi_aligned) * cpi_may_2024

# Prepare data for Prophet
# Prophet requires a DataFrame with columns 'ds' (datetime) and 'y' (value)
prophet_df = retail_sales_may2024_dollars.reset_index()
prophet_df.columns = ['ds', 'y']

# Initialize and fit Prophet model
# Prophet automatically handles seasonality
model = Prophet(seasonality_mode='additive') # or 'multiplicative' depending on the data
model.fit(prophet_df)

# Create a future dataframe for forecasting
future = model.make_future_dataframe(periods=24, freq='M') # Forecast for the next 24 months

# Make predictions
forecast = model.predict(future)

# Plot the forecast
fig1 = model.plot(forecast)
plt.title('Retail Sales Forecast (Adjusted to May 2024 USD)')
plt.xlabel('Date')
plt.ylabel('Value (May 2024 USD)')
plt.show()

# Plot forecast components
fig2 = model.plot_components(forecast)
plt.show()

# Cross-validation to estimate forecast accuracy
# Set the initial training period, horizon, and period for cross-validation
# Example: Initial training data of 5 years, evaluate forecast over 1 year, stepping by 6 months
# Adjust these parameters based on your data history and desired evaluation window
initial_years = 5
horizon_months = 12
period_months = 6

# Calculate initial and period based on data frequency (assuming monthly data)
# Ensure initial is at least 2 * horizon
initial_timedelta = str(initial_years * 365) + ' days' # Prophet expects string format
horizon_timedelta = str(horizon_months) + ' days' # Prophet treats months as ~30.4 days for timedelta
period_timedelta = str(period_months) + ' days'


print(f"Performing cross-validation with initial={initial_timedelta}, horizon={horizon_timedelta}, period={period_timedelta}")

# Ensure initial data is sufficient for cross-validation
if (prophet_df['ds'].max() - prophet_df['ds'].min()).days < 2 * horizon_months * 30.4:
     print("Data history is too short for the chosen cross-validation horizon. Reducing horizon.")
     # Adjust horizon, maybe to half of available data length
     horizon_months = max(3, int((prophet_df['ds'].max() - prophet_df['ds'].min()).days / (2 * 30.4)))
     horizon_timedelta = str(horizon_months) + ' days'
     print(f"Adjusted horizon to {horizon_months} months ({horizon_timedelta}).")
     if (prophet_df['ds'].max() - prophet_df['ds'].min()).days < 2 * horizon_months * 30.4:
         print("Data history still too short. Cannot perform cross-validation with this configuration.")
         # Skip cross-validation if data is severely limited
         cross_validation_results = None
     else:
        try:
            cross_validation_results = cross_validation(model,
                                                        initial=initial_timedelta,
                                                        horizon=horizon_timedelta,
                                                        period=period_timedelta,
                                                        parallel="processes") # Use parallel processing for faster execution
        except Exception as e:
             print(f"Cross-validation failed: {e}")
             print("This might happen if the data is too short or parameters are incompatible.")
             cross_validation_results = None

else:
    try:
        cross_validation_results = cross_validation(model,
                                                    initial=initial_timedelta,
                                                    horizon=horizon_timedelta,
                                                    period=period_timedelta,
                                                    parallel="processes")
    except Exception as e:
         print(f"Cross-validation failed: {e}")
         cross_validation_results = None


# Evaluate performance metrics, including MAPE
if cross_validation_results is not None:
    performance_metrics_df = performance_metrics(cross_validation_results, metrics=['mape'])

    # Print the performance metrics
    print("\nCross-validation Performance Metrics:")
    print(performance_metrics_df)

    # Plot MAPE over the forecast horizon
    fig3 = plot_cross_validation_metric(cross_validation_results, metric='mape')
    plt.title('MAPE over Forecast Horizon')
    plt.xlabel('Horizon (Days)')
    plt.ylabel('MAPE')
    plt.show()

    # Calculate average MAPE
    average_mape = performance_metrics_df['mape'].mean()
    print(f"\nAverage MAPE from cross-validation: {average_mape:.2f}%")

else:
    print("\nCross-validation could not be completed. Cannot estimate forecast accuracy using MAPE.")